In [ ]:
import sys
from pathlib import Path

# project root on sys.path so `horizon` and `eval` resolve from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from horizon.utils.loaders import get_fds
from eval.report import characterize_lazy

In [2]:
# pick a dataset — edit these two paths
fds_path = Path("..") / "datasets" / "service_tickets_21m" / "fds.csv"
data_path = Path("..") / "datasets" / "service_tickets_21m" / "clean.parquet"

In [3]:
import polars as pl

scan = pl.scan_parquet if data_path.suffix.lower() == ".parquet" else pl.scan_csv
print(scan(data_path).collect_schema().names())

['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']


In [4]:
fds = get_fds(fds_path)
print(f"Loaded {len(fds)} FDs")
fds

Loaded 20 FDs


[FD(agency -> agency name),
 FD(agency name -> agency),
 FD(location -> latitude),
 FD(location -> longitude),
 FD(location -> x coordinate (state plane)),
 FD(location -> y coordinate (state plane)),
 FD(latitude, longitude -> location),
 FD(x coordinate (state plane), y coordinate (state plane) -> location),
 FD(community board -> borough),
 FD(bbl -> community board),
 FD(incident zip -> borough),
 FD(bbl -> police precinct),
 FD(bbl -> incident zip),
 FD(police precinct -> borough),
 FD(council district -> borough),
 FD(incident zip -> city),
 FD(bbl -> council district),
 FD(problem detail (formerly descriptor) -> problem (formerly complaint type)),
 FD(bbl -> street name),
 FD(bbl -> incident address)]

In [5]:
# characterize_lazy reads only attr(fd) per FD and prints progress.
result = characterize_lazy(data_path, fds)

[1/20] FD(agency -> agency name)
    loaded 21,164,676 rows, 765.5 MB
    G3=1.323e-06, 2 violation rows
[2/20] FD(agency name -> agency)
    loaded 21,164,676 rows, 765.5 MB
    G3=0, 0 violation rows
[3/20] FD(location -> latitude)
    loaded 21,164,676 rows, 1063.1 MB
    G3=0, 0 violation rows
[4/20] FD(location -> longitude)
    loaded 21,164,676 rows, 1082.9 MB
    G3=0, 0 violation rows
[5/20] FD(location -> x coordinate (state plane))
    loaded 21,164,676 rows, 950.7 MB
    G3=0, 0 violation rows
[6/20] FD(location -> y coordinate (state plane))
    loaded 21,164,676 rows, 926.6 MB
    G3=0, 0 violation rows
[7/20] FD(latitude, longitude -> location)
    loaded 21,164,676 rows, 1358.1 MB
    G3=0, 0 violation rows
[8/20] FD(x coordinate (state plane), y coordinate (state plane) -> location)
    loaded 21,164,676 rows, 1089.3 MB
    G3=0, 0 violation rows
[9/20] FD(community board -> borough)
    loaded 21,164,676 rows, 357.0 MB
    G3=0.0006939, 4 violation rows
[10/20] FD(bbl

In [6]:
# drop FDs whose LHS redundancy is below MIN_REDUNDANCY — too few repeats to
# support a pattern (avg LHS group size ≈ n^MIN_REDUNDANCY). result already
# contains fd_lhs_redundancy, so this is a pure-Python filter on the dict.
import pandas as pd

MIN_REDUNDANCY = 0.1

red = result["fd_lhs_redundancy"]
kept = [fd for fd in fds if red[fd.lhs] >= MIN_REDUNDANCY]
dropped = [fd for fd in fds if red[fd.lhs] < MIN_REDUNDANCY]
print(f"kept {len(kept)} / {len(fds)} FDs; dropping {len(dropped)} (red < {MIN_REDUNDANCY}):")
for fd in dropped:
    print(f"  {red[fd.lhs]:.3f}  {fd}")

result["g3_error"] = {fd: v for fd, v in result["g3_error"].items() if fd in kept}
result["violation_clusters"] = {fd: v for fd, v in result["violation_clusters"].items() if fd in kept}
result["fd_lhs_redundancy"] = {lhs: v for lhs, v in red.items() if any(fd.lhs == lhs for fd in kept)}

pd.DataFrame(
    [{"from": ";".join(fd.lhs), "to": fd.rhs} for fd in kept]
).to_csv(fds_path, index=False)
fds = kept

kept 20 / 20 FDs; dropping 0 (red < 0.1):


In [7]:
from pprint import pprint

summary = {k: v for k, v in result.items() if k not in {"violation_clusters"}}
pprint(summary, sort_dicts=False)

{'attribute_overlap': 0.5714285714285714,
 'fd_interaction_cases': {'IC3', 'IC1', 'IC4', 'IC2'},
 'fd_lhs_redundancy': {('agency',): 0.8195073172702142,
                       ('agency name',): 0.8167494061053839,
                       ('location',): 0.16103105522335773,
                       ('latitude', 'longitude'): 0.1610310145360467,
                       ('x coordinate (state plane)', 'y coordinate (state plane)'): 0.1610310145360467,
                       ('community board',): 0.7416873006136926,
                       ('bbl',): 0.19103882352085436,
                       ('incident zip',): 0.6153227514294237,
                       ('police precinct',): 0.7417151359246517,
                       ('council district',): 0.7654206319854686,
                       ('problem detail (formerly descriptor)',): 0.5745631006214659},
 'g3_error': {FD(agency -> agency name): 1.3229590662966473e-06,
              FD(agency name -> agency): 0.0,
              FD(location -> latitude): 0.

In [32]:
# inspect violations for one FD: pick by index
i = 17
fd = fds[i]
print(f"[{i}] {fd}   G3 = {result['g3_error'][fd]:.4f}")
result["violation_clusters"][fd].head(30)

[17] FD(problem detail (formerly descriptor) -> problem (formerly complaint type))   G3 = 0.0845


problem detail (formerly descriptor),problem (formerly complaint type),count
str,str,u32
"""Animal Waste""","""Animal in a Park""",1110
"""Animal Waste""","""Unsanitary Animal Facility""",287
"""Bag/Wallet""","""Lost Property""",16920
"""Bag/Wallet""","""Found Property""",196
"""Banging/Pounding""","""Noise - Residential""",657880
…,…,…
"""Car Service Company Complaint""","""For Hire Vehicle Complaint""",4798
"""Car Service Company Complaint""","""Dispatched Taxi Complaint""",9
"""Car Service Company Complaint""","""Taxi Complaint""",4
